In [8]:

import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from ultralytics import YOLO


In [18]:
import os
import cv2

In [ ]:
WORKING_DIR = os.getcwd()
PARENT_DIR = os.path.dirname(WORKING_DIR)

VIDEO_PATH = os.path.join(PARENT_DIR, "sample.mp4")
OUTPUT_PATH = os.path.join(WORKING_DIR, "output_imgsz960_model11m.mp4")

print(f"Current working directory: {WORKING_DIR}")
print(f"Video path: {VIDEO_PATH}")
print(f"Output path: {OUTPUT_PATH}")

Current working directory: c:\Users\jetch\OneDrive\Desktop\JJ\2026\Job Application\footfall_cam\testing
Video path: c:\Users\jetch\OneDrive\Desktop\JJ\2026\Job Application\footfall_cam\sample.mp4
Output path: c:\Users\jetch\OneDrive\Desktop\JJ\2026\Job Application\footfall_cam\testing\output_imgsz960_model11m.mp4


In [20]:
# Small pretrained object-detection model.
model = YOLO("yolo11m.pt")

In [21]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}")
print(f"Resolution: {width} x {height}")
print(f"Total frames: {total_frames}")
print(f"Duration: {total_frames / fps:.2f} seconds" if fps > 0 else "Duration unknown")

FPS: 25.0
Resolution: 960 x 720
Total frames: 1341
Duration: 53.64 seconds


In [22]:
writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height),
)

if not writer.isOpened():
    raise RuntimeError(f"Cannot create output: {OUTPUT_PATH}")

try:
    while True:
        success, frame = cap.read()

        if not success:
            break

        # COCO class 0 = person.
        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=0.25,
            imgsz=960,
            verbose=False,
        )

        # Render boxes and track IDs onto the frame.
        annotated_frame = results[0].plot()
        writer.write(annotated_frame)

finally:
    cap.release()
    writer.release()

print(f"Saved overlay video: {OUTPUT_PATH}")

Saved overlay video: c:\Users\jetch\OneDrive\Desktop\JJ\2026\Job Application\footfall_cam\testing\output_imgsz960_model11m.mp4
